# 02 - Baseline: Lokal-saja vs Centralized (TANPA SFM)

**Tujuan belajar:** memahami DUA batas (bound) yang mengapit federated learning,
sebelum kita bangun FedAvg di notebook 03.

- **Centralized (upper bound):** semua data klien digabung di satu tempat, latih satu MLP.
  Ini performa terbaik yang bisa dicapai -- tapi melanggar privasi (data keluar dari klien).
- **Lokal-saja (lower bound):** tiap klien latih MLP sendiri, tanpa berbagi apa pun.
  Privasi terjaga penuh, tapi tiap model cuma lihat sebagian data.

Federated learning (nb03) berusaha mendekati centralized **tanpa** memindahkan data,
hanya berbagi BOBOT. Notebook ini menyiapkan angka pembanding itu.

**Model:** MLP numpy `[d -> 128 -> 64 -> 1]` sigmoid, **identik** dengan
`aws/fed_node_survey.py` (node riset survey) supaya pelajaran ini nyambung ke SOTA tanpa-SFM.

**Prasyarat:** jalankan `01_data_partition.ipynb` dulu (menghasilkan `fl_data/`).
Dijalankan di AWS SageMaker.


In [ ]:
import sys
!{sys.executable} -m pip install -q numpy scikit-learn
print('deps OK')

## 1. MLP numpy (identik fed_node_survey.py)

Arsitektur & matematika SAMA dengan node riset: ReLU hidden, sigmoid output, BCE loss, SGD.
Satu-satunya beda: dimensi input `d` mengikuti jumlah fitur CIC-IDS2017 (dinamis),
bukan 9 (yang di survey khusus cross-dataset).


In [ ]:
import numpy as np
SEED=42; rng=np.random.RandomState(SEED)

def init_weights(d, seed=42):
    r=np.random.RandomState(seed); L=[d,128,64,1]; W=[]
    for a,b in zip(L[:-1],L[1:]):
        W.append(r.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W

def _relu(x): return np.maximum(0,x)
def _sig(x):  return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))

def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; out=_sig(a3).ravel()
    return out,(X,a1,z1,a2,z2,a3,out)

def train_mlp(W,X,y,epochs=5,lr=3e-3,batch=256,seed=42):
    r=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=r.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            out,(X_,a1,z1,a2,z2,a3,o)=forward(W,Xb)
            g=(o-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0); gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0);        gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                W[i]=W[i]-lr*np.asarray(gv).reshape(W[i].shape)
    return W

def predict(W,X): return forward(W,X)[0]

def zscore_fit(X): mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zscore_apply(X,mu,sd): return (X-mu)/sd
print('MLP siap (identik node survey)')

## 2. Muat partisi klien (hasil nb01)


In [ ]:
import os, glob
DATA='./fl_data'
assert os.path.isdir(DATA), 'Jalankan 01_data_partition.ipynb dulu (folder fl_data/ belum ada)'
clients=sorted(set(int(f.split('_')[1]) for f in os.listdir(DATA) if f.startswith('client_')))
print('klien ditemukan:', clients)
def load(split):
    return {i:(np.load(f'{DATA}/client_{i}_{split}.npz')['X'],
               np.load(f'{DATA}/client_{i}_{split}.npz')['y']) for i in clients}
tr=load('train'); te=load('test')
g=np.load(f'{DATA}/global_test.npz'); Xg,yg=g['X'],g['y']
for i in clients:
    print(f'klien {i}: train {len(tr[i][1])}, test {len(te[i][1])}')
print('global_test:', len(yg), 'baris | d =', Xg.shape[1], 'fitur')

## 3. Metrik evaluasi

Pakai MCC (Matthews) + F1 + recall + precision. MCC dipilih karena tahan terhadap
kelas tak-seimbang (attack% < 50%), konsisten dengan paper-paper proyek ini.


In [ ]:
from sklearn.metrics import matthews_corrcoef, f1_score, recall_score, precision_score
def evaluate(W, mu, sd, X, y, thr=0.5):
    p=predict(W, zscore_apply(X,mu,sd)); yp=(p>=thr).astype(int)
    return dict(mcc=matthews_corrcoef(y,yp), f1=f1_score(y,yp,zero_division=0),
               rec=recall_score(y,yp,zero_division=0), prec=precision_score(y,yp,zero_division=0))

## 4. CENTRALIZED (upper bound)

Gabung SEMUA data train klien, fit satu z-score global, latih satu MLP.
Ini skenario 'ideal' yang melanggar privasi -- jadi patokan atas saja.


In [ ]:
Xc=np.vstack([tr[i][0] for i in clients]); yc=np.concatenate([tr[i][1] for i in clients])
mu_c,sd_c=zscore_fit(Xc)
W0=init_weights(Xc.shape[1], SEED)
Wc=train_mlp(W0, zscore_apply(Xc,mu_c,sd_c), yc, epochs=15, lr=3e-3, seed=SEED)
cen=evaluate(Wc, mu_c, sd_c, Xg, yg)
print('CENTRALIZED (global test):', {k:round(v,4) for k,v in cen.items()})

## 5. LOKAL-SAJA (lower bound)

Tiap klien latih MLP pada datanya sendiri (z-score lokal), lalu diuji di global test.
Kita laporkan rata-rata antar klien -- tak ada klien yang lihat data klien lain.


In [ ]:
loc=[]
for i in clients:
    Xi,yi=tr[i]; mu_i,sd_i=zscore_fit(Xi)
    Wi=train_mlp(init_weights(Xi.shape[1],SEED), zscore_apply(Xi,mu_i,sd_i), yi, epochs=15, lr=3e-3, seed=SEED)
    m=evaluate(Wi, mu_i, sd_i, Xg, yg); loc.append(m)
    print(f'  klien {i}: mcc={m["mcc"]:.4f} f1={m["f1"]:.4f}')
local_mean={k:float(np.mean([d[k] for d in loc])) for k in loc[0]}
print('LOKAL-SAJA (rata-rata klien):', {k:round(v,4) for k,v in local_mean.items()})

## 6. Ringkasan: dua bound yang mengapit FL

Simpan ke `learn_out/02_baselines.csv`. Di nb03, FedAvg diharapkan berada
di ANTARA lokal-saja dan centralized -- makin dekat ke centralized makin baik.


In [ ]:
import csv
os.makedirs('learn_out',exist_ok=True)
rows=[('centralized',cen),('local_only_mean',local_mean)]
with open('learn_out/02_baselines.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['method','mcc','f1','recall','precision'])
    for name,m in rows: w.writerow([name,round(m['mcc'],4),round(m['f1'],4),round(m['rec'],4),round(m['prec'],4)])
print('tersimpan: learn_out/02_baselines.csv')
print(f'\nGAP yang harus dikejar FedAvg: {cen["mcc"]-local_mean["mcc"]:+.4f} MCC')
print('Langkah berikut: 03_fedavg.ipynb -- berbagi bobot, bukan data.')